# GRPO from First Principles — Colab quickstart

Runtime → Change runtime type → **T4 GPU**.

Set `REPO_URL` to your copy of the repository, or upload the repository zip and set `ZIP_PATH`.

In [ ]:
REPO_URL = "https://github.com/RamkrushnaSaner/grpo-financial-reasoning.git"
ZIP_PATH = None          # e.g. "/content/grpo-financial-reasoning.zip"

import os, subprocess
if ZIP_PATH:
    subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", "/content"], check=True)
elif not os.path.exists("/content/grpo-financial-reasoning"):
    subprocess.run(["git", "clone", REPO_URL, "/content/grpo-financial-reasoning"], check=True)
%cd /content/grpo-financial-reasoning
!pip install -q -e ".[train,dev]"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Correctness checks (CPU, under a minute)
The test suite (including the full pipeline on a tiny stand-in model), the mathematical validation, and the CPU smoke run.

In [ ]:
!python -m pytest -q
!grpo validate
!grpo run configs/cpu_smoke.yaml --no-probes

## 2. GPU smoke run (~5 minutes)
Every stage on the real model with a handful of steps. The numbers are meaningless; this proves the GPU path works before a long job.

In [ ]:
!grpo run configs/smoke.yaml

## 3. A full run
- `configs/run6.yaml` reproduces the reported run (dataset version 1).
- `configs/base.yaml` is the same method on the corrected dataset.
- `configs/gated.yaml` tests the fix for the format loophole (not yet validated).

To keep results across Colab disconnects, mount Drive and add `--set experiment.output_dir=/content/drive/MyDrive/grpo_results`. Re-running the same command resumes from the cached stages.

In [ ]:
CONFIG = "configs/run6.yaml"
!grpo run {CONFIG}

In [ ]:
from IPython.display import Image, Markdown, display
from grpo.config import load_config
from grpo.experiment import run_dir_for
run_dir = run_dir_for(load_config(CONFIG))   # pass the same --set overrides here if you used any
display(Image(f"{run_dir}/curves.png"))
display(Image(f"{run_dir}/before_after.png"))
display(Markdown(open(f"{run_dir}/report.md").read()))

## 4. Ask one question to the base and the trained model

In [ ]:
!grpo ask {CONFIG} --samples 6 \
    --context "Current annual revenue is \$4.8B with a 6% annual growth rate." \
    --question "What is the projected revenue in \$B for next year?" \
    --answer 5.09 --answer-type dollar

## 5. Optional: reward ablations (~1 hour)
The base config plus three ablations, then a side-by-side table. One seed each: differences under ~15 points at n=60 are not evidence (see docs/TECH_DEBT.md, A2).

In [ ]:
!grpo sweep configs/base.yaml configs/ablations/*.yaml --no-probes